POS Tagging

In [ ]:
# import libraries

import json
import math
import regex as re
import unicodedata
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
from datasets import load_dataset

dataset = load_dataset("masakhane/masakhapos", "yor", verification_mode="basic_checks", trust_remote_code=True)

# Let's look at a training example
print(dataset["train"][0])


{'id': '0', 'tokens': ['Ọ̀gbẹ́ni', 'Nuhu', 'Adam', 'kúrò', 'nípò', 'bí', 'ẹní', 'yọ', 'jìgá'], 'upos': [0, 10, 10, 16, 0, 14, 0, 16, 0]}


In [31]:
upos_labels = masakha_dataset["train"].features["upos"].feature.names
print("UPOS labels:", upos_labels)

UPOS labels: ['NOUN', 'PUNCT', 'ADP', 'NUM', 'SYM', 'SCONJ', 'ADJ', 'PART', 'DET', 'CCONJ', 'PROPN', 'PRON', 'X', '_', 'ADV', 'INTJ', 'VERB', 'AUX']


In [ ]:


import spacy
from spacy.tokens import DocBin
from datasets import load_dataset

# Step 2: Create a blank spaCy pipeline (no pre-loaded model)
nlp = spacy.blank("xx")  # 'xx' = multilingual blank model

tagger = nlp.add_pipe("tagger")

# Register all your POS labels manually
for label in upos_labels:
    tagger.add_label(label)


# Step 5: Convert examples to spaCy DocBin format
doc_bin = DocBin()
for example in masakha_dataset["train"]:
    words = example["tokens"]
    tags = example["upos"]
    doc = nlp.make_doc(" ".join(words))
    for token, tag in zip(doc, tags):
        tag_str = upos_labels[tag]  # Convert tag index to actual string label
        token.pos_ = tag_str
        _ = nlp.vocab.strings.add(tag_str)
    doc_bin.add(doc)

# Step 6: Save to disk
doc_bin.to_disk("yoruba_train.spacy")
print("✅ spaCy training data saved to yoruba_train.spacy")

TypeError: unhashable type: 'list'

In [33]:
!python -m spacy init config config.cfg --lang xx --pipeline tagger --optimize efficiency



✘ The provided output file already exists. To force overwriting the
config file, set the --force or -F flag.



In [39]:
import spacy
from spacy.tokens import DocBin

# Load your training file
doc_bin = DocBin().from_disk("yoruba_train.spacy")
docs = list(doc_bin.get_docs(spacy.blank("xx").vocab))

# Check the first one
for token in docs[0]:
    print(token.text, token.pos_, token.pos)


Ọ̀gbẹ́ni NOUN 92
Nuhu PROPN 96
Adam PROPN 96
kúrò VERB 100
nípò NOUN 92
bí ADV 86
ẹní NOUN 92
yọ VERB 100
jìgá NOUN 92


In [34]:
nlp = spacy.load("yoruba_pos_model/model-best")


In [35]:
def pos_tags(text, model_path="yoruba_pos_model/model-best"):
    nlp = spacy.load(model_path)
    doc = nlp(text)
    return [(token.text, token.tag_) for token in doc]



In [36]:
pos_tags_result = pos_tags("Mo n kọ́ ẹ̀kọ́ nípa èdè Yorùbá.")
print("POS Tags:", pos_tags_result)

POS Tags: [('Mo', '11'), ('n', '17'), ('kọ́', '16'), ('ẹ̀kọ́', '0'), ('nípa', '0'), ('èdè', '0'), ('Yorùbá', '10'), ('.', '1')]


In [37]:
import spacy

# Load the trained model
nlp = spacy.load("yoruba_pos_model/model-best")

# Try it on a test sentence
text = "Mo ń kọ́ ẹ̀kọ́ nipa èdè Yorùbá"
doc = nlp(text)

# Print each token and its POS tag
for token in doc:
    print(f"{token.text:10} → {token.pos_}")


Mo         → 
ń         → 
kọ́        → 
ẹ̀kọ́      → 
nipa       → 
èdè        → 
Yorùbá     → 
